# Global Evaluator Audit

This notebook is the single global inspection notebook for CALE evaluator results. The filename is historical: despite `strong_evaluator_results.ipynb`, this notebook is intended to cover the full current design, including the rule-based full run and the strong-evaluator subsets.

本 notebook 用来把所有结果放在同一个全局检测框架里，而不是只看 strong evaluator。文件名里的 `strong` 是历史遗留，不代表这里只分析 strong evaluator。


This is an audit notebook, not a leaderboard. We intentionally include evaluator backends and protocols with different strengths, including simpler heuristic and direct-judge baselines, to test whether CALE conclusions are robust to the scoring backend and evaluation protocol. A weaker backend/protocol is therefore a control condition or stress-test condition, not an accidental apples-to-oranges comparison.

这是一个 audit notebook，不是排行榜。我们故意把较弱的 heuristic/direct baselines 和 strong LLM judges 放在一起，是为了检查结论是否依赖某个 backend/protocol；弱 evaluator/protocol 在这里是控制条件和压力测试，不是要和 strong evaluator 直接争第一。

It explicitly separates three layers that should not be mixed:

1. **Target model**: the model that generated the candidate response.
2. **Evaluator backend**: the judge implementation/model used to score the response.
3. **Evaluator variant**: the evaluation protocol or CALE variant.

中文速读：`model_name` 是被评价的回答模型，不是 evaluator；`evaluator_backend_*` 是谁在评价；`variant` 是用什么评价结构。

Important naming warning: the rule-based heuristic backend is not Qwen, not Llama, and not an average of Qwen/Llama. It is a non-LLM scoring backend applied to whichever target-model responses are loaded.

重要命名提醒：rule-based heuristic backend 不是 Qwen，不是 Llama，也不是 Qwen/Llama 平均值；它是一个非 LLM 的规则打分 backend，用来评价已经生成好的 target-model responses。

Current evaluator evidence is not “four evaluator models.” It is organized as `evaluator_backend × evaluator_variant/protocol × target_split`: backends are rule-based, Qwen2.5-7B, and DeepSeek; protocols include direct/full/CALE ladder variants; splits include pooled and target-specific views.

当前 evaluator 证据不是“四个模型”的并列比较，而是 `backend × protocol/variant × target_split` 的三层分析结构。表格行数变多，是因为同一个 backend 会被拆成不同 protocol 和 target split。


## 0. Concept Dictionary

| Layer | Meaning | Source in this notebook | Examples |
|---|---|---|---|
| `target_model` | response-generating model being evaluated | copied from CSV `model_name` | Qwen2.5-1.5B, Llama-3.2-1B |
| `evaluator_backend` | judge backend/model that performs scoring | injected from `RUNS` registry | rule-based scoring backend, Qwen2.5-7B HF, DeepSeek V4-Pro |
| `evaluator_variant` | scoring protocol / evaluation design | copied from CSV `variant` | direct_llm_judge / direct_trustllm_heuristic, generic_cale, attack_aware_cale, full_attack_aware_cale |
| `target_split` | analysis scope over target-model responses, not an evaluator | derived from `target_model` | `pooled_qwen_llama`, `target_qwen_only`, `target_llama_only`, `pooled_all_available_targets` |

Do not describe `full_attack_aware_cale` as an evaluator backend. It is a variant/protocol.

不要把 `full_attack_aware_cale` 叫成 evaluator backend；它是 evaluator variant。

If a table shows one value for the rule-based backend without a target split, read it as pooled over available target-response rows, not as an internal model average.

如果表格里 rule-based backend 没有拆 target split，那只是分析时把可用 target-response rows pooled 了，不是 heuristic judge 内部在做模型平均。

Comparisons are interpreted only along their valid audit axis: protocol comparisons should be made within the same backend and matched target responses; backend comparisons should be made on shared rows/variants as agreement or sensitivity checks, not as absolute quality rankings.

比较时要看清比较轴：protocol 比较应在同一个 backend、同一批 target responses 上做；backend 比较应在 shared rows / same variant 上看一致性或敏感性，不要把所有均值摊成一个总排名。

`pooled_qwen_llama` and `pooled_all_available_targets` are analysis scopes, not evaluator backends. `pooled_qwen_llama` means both target-response splits are present; `pooled_all_available_targets` means the table pools whatever target evidence exists and should not be described as balanced target evidence.

`pooled_qwen_llama` 和 `pooled_all_available_targets` 是分析范围，不是 evaluator backend。前者表示 Qwen/Llama target 都在；后者表示只合并当前可用 target evidence，不能写成 balanced target evidence。


## 0.1 What Does `direct` Mean Here?

`direct` means a **single-pass holistic judge baseline**, not an unstructured or information-free evaluation.

这里的 `direct` 指的是**单次整体判断的 baseline**，不是没有标准、也不是不给参考信息。

In the model-based evaluator backends, `direct_llm_judge` uses the prompt built by `build_direct_judge_prompt(...)` in `llm_judge.py`.

在 model-based evaluator backends 里，`direct_llm_judge` 使用的是 `llm_judge.py` 里的 `build_direct_judge_prompt(...)`。

That prompt gives the judge the target model response plus reference information:

- adversarial/user question
- target model response
- reference fact
- reference label
- attack profile
- dataset/domain/risk metadata
- a short holistic evaluation form

But it asks for only one JSON decision:

```json
{"label": "CORRECTED | PARTIALLY_CORRECTED | NOT_CORRECTED | UNKNOWN", "rationale": "..."}
```

It does **not** ask the judge to produce CALE's construct-level checklist, claim-evidence table, per-dimension scores, or structured behavior variables.

它不会要求 judge 输出 CALE 的分维度 checklist、claim-evidence table、每个 construct 的 score，或者完整 behavior variables。

Therefore:

| Variant | Backend type | Meaning |
|---|---|---|
| `direct_llm_judge` | DeepSeek / HF / OpenAI-style LLM judge | model-based holistic direct baseline |
| `direct_trustllm_heuristic` | rule-based scoring backend only | rule-based holistic direct baseline; no LLM call; not equivalent to `direct_llm_judge` |
| `full_attack_aware_cale` | any supported backend | structured CALE variant with construct-level diagnostics |

When this notebook compares direct vs full CALE, it uses the backend-specific direct baseline:

- rule-based backend: `direct_trustllm_heuristic` vs `full_attack_aware_cale`
- Qwen2.5-7B / DeepSeek backends: `direct_llm_judge` vs `full_attack_aware_cale`

所以 direct-vs-CALE 不是在比较“有无参考信息”，而是在比较“整体单标签判断” vs “结构化 CALE 分维度判断”。


In [ ]:
from pathlib import Path
import json
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from analyze_behavior_matrix import numeric_behavior_columns, run_pca
from visualize_behavior_matrix import (
    CORE_BEHAVIOR_COLUMNS,
    PROXY_COLUMNS,
    existing_numeric_columns,
    save_heatmap,
    save_missingness_plot,
    summarize_by,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

OUTDIR = Path('figures/global_evaluator_audit')
OUTDIR.mkdir(parents=True, exist_ok=True)

EXPECTED_TARGET_SPLITS = {'target_qwen_only', 'target_llama_only'}
CALE_VARIANTS = {'generic_cale', 'attack_aware_cale', 'full_attack_aware_cale'}
FULL_CALE_VARIANT = 'full_attack_aware_cale'
MODEL_BASED_DIRECT_VARIANT = 'direct_llm_judge'
HEURISTIC_DIRECT_VARIANT = 'direct_trustllm_heuristic'

VARIANT_ROLE = {
    'baseline_binary': 'baseline',
    'baseline_likert': 'baseline',
    'direct_trustllm_heuristic': 'direct',
    'direct_llm_judge': 'direct',
    'generic_cale': 'cale_family_generic',
    'attack_aware_cale': 'cale_family_attack_aware',
    'full_attack_aware_cale': 'cale_family_full',
}

RUNS = [
    {
        'run_id': 'heuristic_full',
        'evaluator_backend_family': 'heuristic',
        'evaluator_backend_model': 'heuristic/default',
        'evaluator_backend_label': 'Rule-based scoring backend (non-LLM)',
        'behavior_matrix_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_full_eval_behavior_matrix.csv'),
        'report_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_full_eval_report.json'),
        'expected_rows': 239976,
        'expected_variants': ['baseline_binary', 'baseline_likert', 'direct_trustllm_heuristic', 'generic_cale', 'attack_aware_cale', 'full_attack_aware_cale'],
        'expected_target_splits': ['target_qwen_only', 'target_llama_only'],
        'subset_policy': 'full_neutral_fever_qwen_and_llama_targets',
        'direct_baseline_variant': HEURISTIC_DIRECT_VARIANT,
        'notes': 'Main full-sample rule-based result; primary source for PCA/CFA.',
    },
    {
        'run_id': 'qwen25_7b_limit1000',
        'evaluator_backend_family': 'hf_local',
        'evaluator_backend_model': 'Qwen/Qwen2.5-7B-Instruct',
        'evaluator_backend_label': 'Qwen2.5-7B local HF judge',
        'behavior_matrix_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_strong_qwen25_7b_limit1000_eval_behavior_matrix.csv'),
        'report_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_strong_qwen25_7b_limit1000_eval_report.json'),
        'expected_rows': 2000,
        'expected_variants': ['direct_llm_judge', 'full_attack_aware_cale'],
        'expected_target_splits': ['target_qwen_only'],
        'subset_policy': 'limit1000_qwen_target_from_fixed_response_file',
        'direct_baseline_variant': MODEL_BASED_DIRECT_VARIANT,
        'notes': 'Matched Qwen-target strong local evaluator subset; use with qwen25_7b_llama_limit1000 for target-model dominance checks.',
    },
    {
        'run_id': 'qwen25_7b_llama_limit1000',
        'evaluator_backend_family': 'hf_local',
        'evaluator_backend_model': 'Qwen/Qwen2.5-7B-Instruct',
        'evaluator_backend_label': 'Qwen2.5-7B local HF judge',
        'behavior_matrix_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_strong_qwen25_7b_llama_limit1000_eval_behavior_matrix.csv'),
        'report_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_strong_qwen25_7b_llama_limit1000_eval_report.json'),
        'expected_rows': 2000,
        'expected_variants': ['direct_llm_judge', 'full_attack_aware_cale'],
        'expected_target_splits': ['target_llama_only'],
        'subset_policy': 'matched_limit1000_llama_target_from_fixed_response_file',
        'direct_baseline_variant': MODEL_BASED_DIRECT_VARIANT,
        'notes': 'Matched Llama-target strong local evaluator subset; expected once server job finishes.',
    },
    {
        'run_id': 'deepseek_v4_pro_limit1000',
        'evaluator_backend_family': 'api',
        'evaluator_backend_model': 'deepseek-v4-pro',
        'evaluator_backend_label': 'DeepSeek V4-Pro API judge',
        'behavior_matrix_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_deepseek_v4_pro_limit1000_eval_behavior_matrix.csv'),
        'report_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_deepseek_v4_pro_limit1000_eval_report.json'),
        'expected_rows': 2000,
        'expected_variants': ['direct_llm_judge', 'full_attack_aware_cale'],
        'expected_target_splits': ['target_qwen_only'],
        'subset_policy': 'limit1000_qwen_target_from_fixed_response_file',
        'direct_baseline_variant': MODEL_BASED_DIRECT_VARIANT,
        'notes': 'Matched Qwen-target strong API evaluator subset; use with deepseek_v4_pro_llama_limit1000 for target-model dominance checks.',
    },
    {
        'run_id': 'deepseek_v4_pro_llama_limit1000',
        'evaluator_backend_family': 'api',
        'evaluator_backend_model': 'deepseek-v4-pro',
        'evaluator_backend_label': 'DeepSeek V4-Pro API judge',
        'behavior_matrix_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_deepseek_v4_pro_llama_limit1000_eval_behavior_matrix.csv'),
        'report_path': Path('outputs/small_models_all/fever_dev_qwen25_15b_llama32_1b_neutral_deepseek_v4_pro_llama_limit1000_eval_report.json'),
        'expected_rows': 2000,
        'expected_variants': ['direct_llm_judge', 'full_attack_aware_cale'],
        'expected_target_splits': ['target_llama_only'],
        'subset_policy': 'matched_limit1000_llama_target_from_fixed_response_file',
        'direct_baseline_variant': MODEL_BASED_DIRECT_VARIANT,
        'notes': 'Matched Llama-target strong API evaluator subset; expected once server job finishes.',
    },
]

METADATA_REQUIRED = ['id', 'model_name', 'variant', 'reference_label', 'dataset']
OUTCOME_COLUMNS = ['final_score', 'quality_label', 'uncertainty']
BEHAVIOR_COLUMNS = CORE_BEHAVIOR_COLUMNS + PROXY_COLUMNS

run_registry = pd.DataFrame([{k: str(v) if isinstance(v, Path) else v for k, v in run.items()} for run in RUNS])
run_registry.to_csv(OUTDIR / 'run_registry.csv', index=False)
run_registry

## 0.2 Current Experimental Design Matrix

This notebook treats the project as a three-layer design, not as a flat list of evaluators.

这个 notebook 把实验看成三层设计，而不是把所有名字混成一串 evaluator。

Current design summary:

- **2 target models**: Qwen2.5-1.5B and Llama-3.2-1B generate responses.
- **3 evaluator backends**: rule-based scoring backend, Qwen2.5-7B local HF judge, DeepSeek V4-Pro API judge.
- **Several evaluator variants/protocols**: baselines, direct holistic baselines, and CALE-family variants.

The rule-based backend is included because it is the full-sample measurement baseline. It is not a strong LLM evaluator and not a target-model average.

rule-based backend 会被纳入，是因为它是当前 full-sample measurement baseline；它不是 strong LLM evaluator，也不是 target-model 平均值。

The design is intentionally heterogeneous. The rule-based backend gives full-sample coverage, strong LLM backends test whether conclusions survive model-based judging, and simpler/direct protocols test whether CALE's structured protocol adds value over less structured scoring.

这个设计本来就是异质的：rule-based backend 提供 full-sample 参照，strong LLM judge 提供 model-based robustness check，direct/generic/partial protocols 提供 protocol ablation 和 sensitivity audit。

Do not count target models, protocols, or splits as evaluator backends. The current backend count is three: rule-based, Qwen2.5-7B judge, and DeepSeek judge. Tables may have more rows because each backend is crossed with protocol variants and target splits.

不要把 target model、protocol、split 也算成 evaluator backend。目前 backend 是 3 类：rule-based、Qwen2.5-7B judge、DeepSeek judge。表格行数更多，是因为 backend 和 protocol/target split 做了交叉。


In [ ]:
design_overview_rows = []
for run in RUNS:
    design_overview_rows.append({
        'run_id': run['run_id'],
        'target_splits_planned': ', '.join(run['expected_target_splits']),
        'evaluator_backend_label': run['evaluator_backend_label'],
        'evaluator_backend_model': run['evaluator_backend_model'],
        'evaluator_variants_planned': ', '.join(run['expected_variants']),
        'direct_baseline_variant_for_this_backend': run['direct_baseline_variant'],
        'subset_policy': run['subset_policy'],
        'notes': run['notes'],
    })

design_overview = pd.DataFrame(design_overview_rows)
design_overview.to_csv(OUTDIR / 'current_experimental_design_matrix.csv', index=False)
display(design_overview)

backend_count = design_overview[['evaluator_backend_model', 'evaluator_backend_label']].drop_duplicates().shape[0]
print(f'Current design: 2 target models, {backend_count} evaluator backends, and backend-specific evaluator variants/protocols.')


## 0.3 Recommended Outputs To Inspect

All generated audit outputs are written to `figures/global_evaluator_audit/` relative to the notebook execution directory. On the server, run this notebook from the CALE project root so relative paths resolve to `outputs/...` and `figures/...`.

建议优先看这些输出：

- `global_interpretation_notes.md`: compact paper-facing summary draft.
- `global_file_audit.csv`: which input files exist, row counts, report availability, and target coverage.
- `current_experimental_design_matrix.csv`: current target/backend/variant design registry.
- `layer_dictionary.csv`: definitions of target model, evaluator backend, evaluator variant, and direct baseline names.
- `global_validation_dashboard.csv`: checks for naming confusion and duplicate row keys.
- `cross_product_coverage_compact.csv`: design coverage by run, target split, backend, and variant.
- `outcome_summary_three_layers.csv`: descriptive score summary by run/backend/target/variant.
- `direct_vs_full_cale_delta.csv`: backend-specific direct baseline versus full CALE.
- `full_cale_construct_profile_by_backend_protocol_target_split.csv`: full CALE construct means split into pooled and target-specific views.
- `full_cale_construct_profile_compact.csv`: notebook-friendly pivot for pooled/Qwen-target/Llama-target construct profiles.
- `backend_pair_score_alignment_by_target_split.csv`: backend agreement split by pooled and target-specific scopes.
- `target_model_dominance_pca_summary.csv`: backend-level PCA summaries for pooled and target-specific full CALE rows.
- `protocol_backend_pca_summary.csv`: PCA variance audit across weaker baselines, direct baselines, and CALE-family protocols.
- `protocol_backend_pca_variance_pooled.png`: PC1 and PC1-PC4 visualization for pooled protocol/backend comparisons.
- `target_model_dominance_pc1_variance.png`: PC1 variance by backend and target split for `full_attack_aware_cale`.
- `heuristic_full_variant_ladder_by_target_split.csv`: heuristic full-run CALE-family ladder.


## 1. Helper Functions

These helpers enforce the layer separation. In particular, `target_model` is copied from `model_name`, while evaluator backend columns are injected from the run registry.

这些 helper 的作用是强制分清三层：target model 来自 CSV，evaluator backend 来自 notebook registry。

In [ ]:
def target_split_name(model_name):
    value = str(model_name).lower()
    if 'qwen' in value:
        return 'target_qwen_only'
    if 'llama' in value:
        return 'target_llama_only'
    return 'target_other'


def target_coverage_label(splits):
    present = set(splits)
    if EXPECTED_TARGET_SPLITS <= present:
        return 'pooled_qwen_llama'
    if present == {'target_qwen_only'}:
        return 'qwen_target_only'
    if present == {'target_llama_only'}:
        return 'llama_target_only'
    if not present:
        return 'missing_file_or_no_target_model'
    return 'partial_or_unknown'


def existing_behavior_columns(df):
    return existing_numeric_columns(df, CORE_BEHAVIOR_COLUMNS) + existing_numeric_columns(df, PROXY_COLUMNS)


def load_run(run):
    path = run['behavior_matrix_path']
    if not path.exists():
        return None
    df = pd.read_csv(path)
    df = df.copy()
    df['target_model'] = df['model_name'].astype(str) if 'model_name' in df.columns else 'unknown_target_model'
    df['target_split'] = df['target_model'].map(target_split_name)
    df['evaluator_variant'] = df['variant'].astype(str) if 'variant' in df.columns else 'unknown_variant'
    df['variant_role'] = df['evaluator_variant'].map(VARIANT_ROLE).fillna('other_or_unknown')
    for key in ['run_id', 'evaluator_backend_family', 'evaluator_backend_model', 'evaluator_backend_label', 'subset_policy', 'direct_baseline_variant']:
        df[key] = run[key]
    return df


def score_column(df):
    if 'final_score' in df.columns:
        return 'final_score'
    if 'score' in df.columns:
        return 'score'
    return None


def short_backend_label(value):
    text = str(value)
    if 'DeepSeek' in text or 'deepseek' in text:
        return 'DeepSeek V4-Pro'
    if 'Qwen2.5-7B' in text or 'Qwen/Qwen2.5-7B' in text:
        return 'Qwen2.5-7B judge'
    if 'Rule-based' in text or 'heuristic/default' in text:
        return 'Rule-based heuristic'
    if 'target_qwen_only' in text:
        return 'Target: Qwen'
    if 'target_llama_only' in text:
        return 'Target: Llama'
    if 'pooled_qwen_llama' in text:
        return 'Pooled Qwen+Llama'
    if 'pooled_all_available_targets' in text:
        return 'Pooled available'
    return text


def short_variant_label(value):
    mapping = {
        'baseline_binary': 'binary baseline',
        'baseline_likert': 'likert baseline',
        'direct_trustllm_heuristic': 'direct heuristic',
        'direct_llm_judge': 'direct LLM',
        'generic_cale': 'generic CALE',
        'attack_aware_cale': 'attack-aware CALE',
        'full_attack_aware_cale': 'full CALE',
    }
    return mapping.get(str(value), str(value))


def short_plot_label(value):
    if isinstance(value, tuple):
        parts = [str(v) for v in value if pd.notna(v)]
        backend = short_backend_label(parts[0]) if parts else ''
        variant = next((short_variant_label(v) for v in parts if str(v) in VARIANT_ROLE), '')
        return f'{backend} | {variant}' if variant else backend
    return short_backend_label(value)


def save_bar(table, path, title, ylabel='value', rotation=25):
    if table.empty:
        return
    labels = [short_plot_label(idx) for idx in table.index]
    long_labels = any(len(label) > 28 for label in labels) or isinstance(table.index, pd.MultiIndex)
    if long_labels:
        fig, ax = plt.subplots(figsize=(10, max(4.8, 0.48 * len(table.index))))
        table = table.copy()
        table.index = labels
        table.plot(kind='barh', ax=ax)
        ax.set_title(title)
        ax.set_xlabel(ylabel)
        ax.set_ylabel('')
        ax.tick_params(axis='y', labelsize=8)
        ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=8)
    else:
        fig, ax = plt.subplots(figsize=(max(7, 0.65 * len(table.index)), 4.8))
        table.plot(kind='bar', ax=ax)
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=rotation)
        ax.legend(loc='best', fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=240, bbox_inches='tight')
    plt.show()


def save_annotated_heatmap(table, path, title, fmt='.3f', vmin=None, vmax=None, cmap='viridis'):
    if table.empty:
        return
    numeric = table.apply(pd.to_numeric, errors='coerce')
    if numeric.dropna(how='all').empty:
        return
    numeric = numeric.dropna(how='all').dropna(axis=1, how='all')
    row_labels = [short_plot_label(idx) for idx in numeric.index]
    col_labels = [short_backend_label(col) for col in numeric.columns]
    fig, ax = plt.subplots(figsize=(max(7, 1.8 * len(col_labels)), max(3.8, 0.55 * len(row_labels))))
    image = ax.imshow(numeric.values, aspect='auto', vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_title(title, pad=12)
    ax.set_xticks(np.arange(len(col_labels)))
    ax.set_xticklabels(col_labels, rotation=0, ha='center', fontsize=9)
    ax.set_yticks(np.arange(len(row_labels)))
    ax.set_yticklabels(row_labels, fontsize=9)
    for i in range(numeric.shape[0]):
        for j in range(numeric.shape[1]):
            value = numeric.iat[i, j]
            if pd.notna(value):
                ax.text(j, i, format(value, fmt), ha='center', va='center', color='white' if value >= numeric.stack().median() else 'black', fontsize=8)
    fig.colorbar(image, ax=ax, fraction=0.035, pad=0.02)
    fig.tight_layout()
    fig.savefig(path, dpi=240, bbox_inches='tight')
    plt.show()


def top_loading_records(loadings, run_id, backend_label, target_split='pooled_all_available_targets', n=8):
    rows = []
    for component in loadings.columns:
        ordered = loadings[component].sort_values(key=lambda s: s.abs(), ascending=False)
        for rank, (variable, loading) in enumerate(ordered.head(n).items(), start=1):
            rows.append({
                'run_id': run_id,
                'evaluator_backend_label': backend_label,
                'target_split': target_split,
                'component': component,
                'rank': rank,
                'variable': variable,
                'loading': loading,
                'absolute_loading': abs(loading),
            })
    return rows


def run_pca_summary(df, run_id, backend_label, target_split, n_components=4, max_missing_share=0.5):
    columns = numeric_behavior_columns(df, include_final_score=False)
    selected = df[columns].apply(pd.to_numeric, errors='coerce') if columns else pd.DataFrame()
    loadings, variance, scores = run_pca(selected, n_components=n_components, max_missing_share=max_missing_share)
    return {
        'run_id': run_id,
        'evaluator_backend_label': backend_label,
        'target_split': target_split,
        'rows': len(df),
        'columns_used': ', '.join(loadings.index.tolist()),
        'n_columns_used': len(loadings.index),
        'pc1_variance': variance.loc[variance['component'].eq('PC1'), 'explained_variance_ratio'].iloc[0] if 'PC1' in set(variance['component']) else np.nan,
        'pc1_pc4_cumulative_variance': variance['explained_variance_ratio'].sum(),
        'pc1_top_variables': ', '.join(loadings['PC1'].abs().sort_values(ascending=False).head(4).index.tolist()) if 'PC1' in loadings else '',
        'pc2_top_variables': ', '.join(loadings['PC2'].abs().sort_values(ascending=False).head(4).index.tolist()) if 'PC2' in loadings else '',
    }, loadings, variance, scores


def paired_direct_vs_cale(df):
    score_col = score_column(df)
    if score_col is None or df.empty:
        return pd.DataFrame()
    direct_variant = str(df.get('direct_baseline_variant', pd.Series([MODEL_BASED_DIRECT_VARIANT])).iloc[0])
    needed = [direct_variant, FULL_CALE_VARIANT]
    subset = df[df['evaluator_variant'].isin(needed)].copy()
    if subset.empty:
        return pd.DataFrame()
    key_cols = ['run_id', 'evaluator_backend_label', 'target_model', 'id']
    if 'dataset' in subset.columns:
        key_cols.insert(0, 'dataset')
    if not all(col in subset.columns for col in key_cols):
        return pd.DataFrame()
    wide = subset.pivot_table(index=key_cols, columns='evaluator_variant', values=score_col, aggfunc='mean')
    if not all(col in wide.columns for col in needed):
        return pd.DataFrame()
    wide = wide.dropna(subset=needed).reset_index()
    wide['direct_baseline_variant'] = direct_variant
    wide['full_minus_direct_score'] = wide[FULL_CALE_VARIANT] - wide[direct_variant]
    wide['direct_baseline_score'] = wide[direct_variant]
    return wide


## 2. File, Provenance, And Coverage Audit

This is the first table to inspect on the server. It tells us which files exist, whether the row counts match, which target models are covered, and whether target splits are missing.

服务器上先看这一张表：它回答文件是否存在、行数是否对、覆盖了哪些 target models、有没有缺 target split。

In [ ]:
matrices = {}
audit_rows = []

for run in RUNS:
    df = load_run(run)
    if df is not None:
        matrices[run['run_id']] = df
        target_models = sorted(df['target_model'].dropna().astype(str).unique())
        target_splits = sorted(df['target_split'].dropna().astype(str).unique())
        variants = sorted(df['evaluator_variant'].dropna().astype(str).unique())
        missing_splits = sorted(set(run['expected_target_splits']) - set(target_splits))
        missing_variants = sorted(set(run['expected_variants']) - set(variants))
    else:
        target_models, target_splits, variants, missing_splits, missing_variants = [], [], [], list(run['expected_target_splits']), list(run['expected_variants'])

    report_path = run.get('report_path')
    audit_rows.append({
        'run_id': run['run_id'],
        'evaluator_backend_label': run['evaluator_backend_label'],
        'evaluator_backend_family': run['evaluator_backend_family'],
        'evaluator_backend_model': run['evaluator_backend_model'],
        'path': str(run['behavior_matrix_path']),
        'report_path': str(report_path) if report_path else '',
        'exists': run['behavior_matrix_path'].exists(),
        'report_path_exists': report_path.exists() if report_path else False,
        'rows': len(df) if df is not None else 0,
        'expected_rows': run['expected_rows'],
        'row_count_ok': (len(df) == run['expected_rows']) if df is not None else False,
        'evaluator_variants': ', '.join(variants),
        'missing_expected_variants': ', '.join(missing_variants),
        'target_models': ', '.join(target_models),
        'target_splits_present': ', '.join(target_splits),
        'target_splits_missing': ', '.join(missing_splits),
        'target_coverage': target_coverage_label(target_splits),
        'coverage_warning': bool(missing_splits),
        'subset_policy': run['subset_policy'],
        'notes': run['notes'],
    })

audit = pd.DataFrame(audit_rows)
audit.to_csv(OUTDIR / 'global_file_audit.csv', index=False)
audit


## 3. Layer Dictionary And Validation Dashboard

These checks are designed to catch naming mistakes such as treating `full_attack_aware_cale` as a backend or treating `Qwen/Qwen2.5-1.5B-Instruct` as a judge.

这些 check 用来防止最容易犯的命名错误：把 full CALE 当 backend，或者把 target Qwen 当 evaluator。

In [ ]:
layer_dictionary = pd.DataFrame([
    {
        'layer': 'target_model',
        'meaning': 'model that generated the candidate response',
        'source_column': 'model_name copied to target_model',
        'examples': 'Qwen/Qwen2.5-1.5B-Instruct; meta-llama/Llama-3.2-1B-Instruct',
        'do_not_mix_with': 'evaluator_backend_model',
    },
    {
        'layer': 'evaluator_backend',
        'meaning': 'judge backend or judge model used to score the response',
        'source_column': 'injected from RUNS registry',
        'examples': 'heuristic/default; Qwen/Qwen2.5-7B-Instruct; deepseek-v4-pro',
        'do_not_mix_with': 'target_model or evaluator_variant',
    },
    {
        'layer': 'evaluator_variant',
        'meaning': 'evaluation protocol or scoring structure',
        'source_column': 'variant copied to evaluator_variant',
        'examples': 'direct_llm_judge; direct_trustllm_heuristic; generic_cale; attack_aware_cale; full_attack_aware_cale',
        'do_not_mix_with': 'evaluator_backend_model',
    },
    {
        'layer': 'variant_role',
        'meaning': 'normalized family role used for dashboard grouping',
        'source_column': 'derived from evaluator_variant via VARIANT_ROLE',
        'examples': 'direct; baseline; cale_family_generic; cale_family_attack_aware; cale_family_full',
        'do_not_mix_with': 'evaluator_backend_model or target_model',
    },
    {
        'layer': 'direct_trustllm_heuristic',
        'meaning': 'rule-based direct baseline for the heuristic backend only; no LLM call',
        'source_column': 'evaluator_variant',
        'examples': 'compare with full_attack_aware_cale only within heuristic/default backend',
        'do_not_mix_with': 'direct_llm_judge or target-model averaging',
    },
    {
        'layer': 'direct_llm_judge',
        'meaning': 'model/API direct baseline using build_direct_judge_prompt(...)',
        'source_column': 'evaluator_variant',
        'examples': 'DeepSeek V4-Pro direct judge; Qwen2.5-7B direct judge',
        'do_not_mix_with': 'direct_trustllm_heuristic',
    },
])
layer_dictionary.to_csv(OUTDIR / 'layer_dictionary.csv', index=False)

validation_rows = []
for run_id, df in matrices.items():
    required_cols = METADATA_REQUIRED + ['evaluator_backend_label', 'evaluator_backend_model', 'evaluator_variant', 'target_model', 'target_split']
    missing_required = [col for col in required_cols if col not in df.columns]
    key_cols = ['run_id', 'dataset', 'id', 'target_model', 'evaluator_backend_model', 'evaluator_variant']
    if all(col in df.columns for col in key_cols):
        duplicate_count = int(df.duplicated(subset=key_cols).sum())
    else:
        duplicate_count = np.nan
    variant_values = set(df['evaluator_variant'].dropna().astype(str)) if 'evaluator_variant' in df.columns else set()
    backend_like_variants = sorted([v for v in variant_values if any(token in v.lower() for token in ['deepseek', 'qwen2.5-7b', 'heuristic full evaluator'])])
    target_models = set(df['target_model'].dropna().astype(str)) if 'target_model' in df.columns else set()
    backend_model = str(df['evaluator_backend_model'].iloc[0]) if 'evaluator_backend_model' in df.columns and len(df) else ''
    backend_in_target = backend_model in target_models

    validation_rows.append({
        'run_id': run_id,
        'status': 'pass' if not missing_required and duplicate_count == 0 and not backend_like_variants and not backend_in_target else 'review',
        'missing_required_columns': ', '.join(missing_required),
        'duplicate_row_key_count': duplicate_count,
        'backend_like_variant_values': ', '.join(backend_like_variants),
        'evaluator_backend_model_appears_as_target_model': backend_in_target,
        'message': 'Review any non-pass row before interpreting global comparisons.',
    })

validation_dashboard = pd.DataFrame(validation_rows)
validation_dashboard.to_csv(OUTDIR / 'global_validation_dashboard.csv', index=False)
display(layer_dictionary)
validation_dashboard

## 4. Schema And Missingness Audit

This checks which behavior variables are available for each run. Direct judge rows usually lack CALE construct subscores, so construct-level analyses must filter to CALE variants.

这里检查每个 run 有哪些 behavior variables。direct judge 通常没有 CALE construct subscores，所以 construct-level 分析必须过滤 CALE variants。

The detailed CSV keeps `target_split` so the full heuristic run does not silently pool Qwen-target and Llama-target rows.

详细版 CSV 会保留 `target_split`，避免 heuristic full run 把 Qwen-target 和 Llama-target 的 missingness 悄悄合并。


In [ ]:
missingness_rows = []
for run_id, df in matrices.items():
    split_groups = [('all_available_targets', df)]
    if 'target_split' in df.columns:
        split_groups.extend(list(df.groupby('target_split', dropna=False)))
    for target_split, split_df in split_groups:
        for col in METADATA_REQUIRED + OUTCOME_COLUMNS + BEHAVIOR_COLUMNS:
            missingness_rows.append({
                'run_id': run_id,
                'evaluator_backend_label': df['evaluator_backend_label'].iloc[0],
                'evaluator_backend_model': df['evaluator_backend_model'].iloc[0] if 'evaluator_backend_model' in df.columns else '',
                'target_split': target_split,
                'column': col,
                'exists': col in df.columns,
                'rows': len(split_df),
                'non_null_share': pd.to_numeric(split_df[col], errors='coerce').notna().mean() if col in df.columns and col in BEHAVIOR_COLUMNS + ['final_score', 'uncertainty'] else (split_df[col].notna().mean() if col in df.columns else 0.0),
            })

schema_missingness = pd.DataFrame(missingness_rows)
schema_missingness.to_csv(OUTDIR / 'schema_missingness_audit.csv', index=False)
schema_missingness.to_csv(OUTDIR / 'schema_missingness_by_run_target_split.csv', index=False)

if not schema_missingness.empty:
    availability = schema_missingness.pivot_table(
        index=['run_id', 'evaluator_backend_label', 'target_split'],
        columns='column',
        values='non_null_share',
        aggfunc='mean',
    ).fillna(0)
    availability.to_csv(OUTDIR / 'schema_availability_pivot.csv')
    availability.to_csv(OUTDIR / 'schema_availability_by_run_target_split.csv')
    behavior_availability = availability[[c for c in BEHAVIOR_COLUMNS if c in availability.columns]]
    if not behavior_availability.empty:
        save_heatmap(behavior_availability, OUTDIR / 'behavior_variable_availability_by_run_target_split.png', 'Behavior Variable Availability by Run and Target Split', vmin=0, vmax=1)
    display(availability.round(2))
else:
    print('No matrices loaded yet.')


## 5. Cross-Product Coverage

This is a **design coverage table**, not a score table.

这是一张**实验覆盖表**，不是结果分数表。

Read it as: for each `target_model × evaluator_backend × evaluator_variant`, how many rows exist?

读法是：每一个 `target_model × evaluator_backend × evaluator_variant` 组合到底有没有跑、跑了多少行。

Important interpretation:

- `available_complete` means this planned combination exists in the current outputs.
- `expected_missing` means this combination is in the planned design but missing from current outputs.
- `not_in_design` means this combination was not part of the planned run.
- `unexpected_available` means a combination appears even though it was not listed in the design registry.
- `rows = 0` does **not** mean the evaluator performed badly.
- Strong evaluators currently run only `direct_llm_judge` and `full_attack_aware_cale`, so their baseline/generic/attack-aware cells are expected to be missing.
- The heuristic backend is the only current run with the full CALE ladder and baselines.

重要解释：

- `available_complete` 表示这个计划内组合已经有输出。
- `expected_missing` 表示这个组合理论上该有，但当前输出里没有。
- `not_in_design` 表示这个组合本来就不在这次实验设计里。
- `unexpected_available` 表示输出里出现了 registry 没列的组合。
- `rows = 0` 不是模型表现差。
- strong evaluator 目前只跑了 `direct_llm_judge` 和 `full_attack_aware_cale`，所以 baseline/generic/attack-aware 缺失是预期的。
- heuristic backend 才是当前唯一包含完整 CALE ladder 和 baselines 的 run。


In [ ]:
combined = pd.concat(matrices.values(), ignore_index=True, sort=False) if matrices else pd.DataFrame()
if not combined.empty:
    combined.to_csv(OUTDIR / 'combined_available_behavior_matrix.csv', index=False)

    observed = combined.groupby(
        ['run_id', 'evaluator_backend_label', 'target_split', 'evaluator_variant', 'variant_role'],
        dropna=False,
    ).agg(
        rows=('id', 'size'),
        target_models=('target_model', lambda s: ', '.join(sorted(s.dropna().astype(str).unique()))),
    ).reset_index()
    observed.to_csv(OUTDIR / 'cross_product_coverage_observed_long.csv', index=False)

    design_rows = []
    for run in RUNS:
        for target_split in run['expected_target_splits']:
            for variant in run['expected_variants']:
                design_rows.append({
                    'run_id': run['run_id'],
                    'evaluator_backend_label': run['evaluator_backend_label'],
                    'target_split': target_split,
                    'evaluator_variant': variant,
                    'variant_role': VARIANT_ROLE.get(variant, 'other_or_unknown'),
                    'design_status': 'in_design',
                    'subset_policy': run['subset_policy'],
                    'notes': run['notes'],
                })
    design_grid = pd.DataFrame(design_rows)

    coverage_full = design_grid.merge(
        observed,
        on=['run_id', 'evaluator_backend_label', 'target_split', 'evaluator_variant', 'variant_role'],
        how='outer',
    )
    coverage_full['rows'] = coverage_full['rows'].fillna(0).astype(int)
    coverage_full['design_status'] = coverage_full['design_status'].fillna('not_in_design')
    coverage_full['coverage_status'] = np.select(
        [
            coverage_full['rows'].gt(0) & coverage_full['design_status'].eq('in_design'),
            coverage_full['rows'].eq(0) & coverage_full['design_status'].eq('in_design'),
            coverage_full['rows'].gt(0) & coverage_full['design_status'].eq('not_in_design'),
        ],
        ['available_complete', 'expected_missing', 'unexpected_available'],
        default='not_in_design',
    )
    coverage_full['target_models'] = coverage_full['target_models'].fillna('')
    coverage_full.to_csv(OUTDIR / 'cross_product_coverage_long.csv', index=False)

    coverage_readable = coverage_full[
        ['run_id', 'evaluator_backend_label', 'target_split', 'evaluator_variant', 'variant_role', 'rows', 'coverage_status', 'subset_policy']
    ].sort_values(['evaluator_backend_label', 'target_split', 'variant_role', 'evaluator_variant'])

    coverage_compact = coverage_readable.copy()
    coverage_compact['cell'] = coverage_compact['coverage_status'] + ' (' + coverage_compact['rows'].astype(str) + ')'
    coverage_compact = coverage_compact.pivot_table(
        index=['run_id', 'evaluator_backend_label', 'target_split'],
        columns='evaluator_variant',
        values='cell',
        aggfunc='first',
        fill_value='not_in_design (0)',
    )
    coverage_compact.to_csv(OUTDIR / 'cross_product_coverage_compact.csv')

    print('Compact design coverage: cells show status plus row count, not performance.')
    display(coverage_compact)
    print('Long design coverage: inspect subset policy and run id here.')
    display(coverage_readable)

    coverage_pivot = coverage_full.pivot_table(
        index=['run_id', 'evaluator_backend_label'],
        columns=['target_split', 'evaluator_variant'],
        values='rows',
        fill_value=0,
        aggfunc='sum',
    )
    coverage_pivot.to_csv(OUTDIR / 'cross_product_coverage_pivot.csv')
else:
    coverage = pd.DataFrame()
    coverage_full = pd.DataFrame()
    coverage_readable = pd.DataFrame()
    coverage_pivot = pd.DataFrame()
    print('No matrices loaded yet. Run this notebook on the server after outputs are available.')


## 6. Outcome Summary By Three Layers

Outcome-level summaries can include direct judge and CALE variants. Backend-to-backend absolute scores are descriptive because scoring scales may differ.

Outcome 层可以比较 direct judge 和 CALE variants；但不同 backend 的绝对分数不是 calibrated leaderboard。

In [ ]:
outcome_rows = []
if not combined.empty:
    sc = score_column(combined)
    group_cols = ['run_id', 'evaluator_backend_label', 'subset_policy', 'target_model', 'target_split', 'evaluator_variant']
    for keys, group in combined.groupby(group_cols, dropna=False):
        run_id, backend_label, subset_policy, target_model, target_split, variant = keys
        row = {
            'run_id': run_id,
            'evaluator_backend_label': backend_label,
            'subset_policy': subset_policy,
            'target_model': target_model,
            'target_split': target_split,
            'evaluator_variant': variant,
            'rows': len(group),
        }
        if sc:
            scores = pd.to_numeric(group[sc], errors='coerce')
            row['n_non_null_score'] = int(scores.notna().sum())
            row['score_missing_share'] = float(scores.isna().mean())
            row['mean_final_score'] = scores.mean()
            row['std_final_score'] = scores.std()
        if 'uncertainty' in group.columns:
            uncertainty = pd.to_numeric(group['uncertainty'], errors='coerce')
            row['n_non_null_uncertainty'] = int(uncertainty.notna().sum())
            row['mean_uncertainty'] = uncertainty.mean()
        if 'quality_label' in group.columns:
            labels = group['quality_label']
            row['quality_label_missing_share'] = float(labels.isna().mean())
            row['quality_label_distribution'] = labels.dropna().astype(str).value_counts(normalize=True).round(3).to_dict()
        outcome_rows.append(row)

outcome_summary = pd.DataFrame(outcome_rows)
outcome_summary.to_csv(OUTDIR / 'outcome_summary_three_layers.csv', index=False)
outcome_summary


In [ ]:
if not outcome_summary.empty and 'mean_final_score' in outcome_summary.columns:
    score_pivot = outcome_summary.pivot_table(
        index=['run_id', 'evaluator_backend_label', 'target_split', 'target_model'],
        columns='evaluator_variant',
        values='mean_final_score',
        aggfunc='mean',
    )
    score_pivot.to_csv(OUTDIR / 'mean_score_by_run_backend_target_variant.csv')
    save_bar(score_pivot, OUTDIR / 'mean_score_by_run_backend_target_variant.png', 'Mean Score by Run, Evaluator Backend, Target, and Variant', ylabel='mean final score')
    print('Descriptive means only. Rows are run x evaluator backend x target model. NaN means this backend/variant/target combination was not run or not in design, not poor performance.')
    display(score_pivot.round(3))


## 7. Direct Judge Versus Full CALE Delta

This section compares the backend-specific direct baseline against `full_attack_aware_cale` on shared ids.

这一节比较的是同一个 backend、同一个 target model 内，backend-specific direct baseline 和 `full_attack_aware_cale` 的差异。

Important: the direct baseline name depends on backend:

- heuristic backend uses `direct_trustllm_heuristic`
- model-based backends use `direct_llm_judge`

`direct` here means holistic single-label judging with reference information, not CALE's structured construct-level evaluation.

这里的 `direct` 是“给参考信息后的整体单标签判断”，不是 CALE 的结构化分维度评价。


In [ ]:
paired_tables = []
for run_id, df in matrices.items():
    paired = paired_direct_vs_cale(df)
    if not paired.empty:
        paired_tables.append(paired)

paired_delta_rows = []
if paired_tables:
    paired_all = pd.concat(paired_tables, ignore_index=True)
    paired_all.to_csv(OUTDIR / 'paired_direct_vs_full_cale_rows.csv', index=False)
    for keys, group in paired_all.groupby(['run_id', 'evaluator_backend_label', 'target_model'], dropna=False):
        run_id, backend_label, target_model = keys
        paired_delta_rows.append({
            'run_id': run_id,
            'evaluator_backend_label': backend_label,
            'target_model': target_model,
            'n_shared_ids': len(group),
            'direct_baseline_variant': group['direct_baseline_variant'].iloc[0],
            'direct_mean_score': group['direct_baseline_score'].mean(),
            'full_cale_mean_score': group[FULL_CALE_VARIANT].mean(),
            'full_minus_direct_mean_score': group['full_minus_direct_score'].mean(),
        })

direct_vs_full_delta = pd.DataFrame(paired_delta_rows)
direct_vs_full_delta.to_csv(OUTDIR / 'direct_vs_full_cale_delta.csv', index=False)
direct_vs_full_delta

## 8. Construct Profile By Backend, Target, And Variant

Construct profiles use CALE behavior columns. Direct judge rows are excluded when they lack construct variables.

Construct profile 使用 CALE behavior columns；没有 construct variables 的 direct judge rows 不进入这一步。

The display table is intentionally split into pooled and target-specific views. Rows are evaluator backend/protocol; columns are target split × construct metric. This prevents the pooled row from hiding whether Qwen-target or Llama-target responses are driving the pattern.

下面的表会拆成 pooled 和 target-specific views。行是 evaluator backend/protocol，列是 target split × construct metric。这样 pooled 结果不会遮住到底是 Qwen-target 还是 Llama-target 在驱动差异。


In [ ]:
profile_rows = []
if not combined.empty:
    cols = existing_behavior_columns(combined)
    cale_like = combined[combined['evaluator_variant'].isin(CALE_VARIANTS)].copy()
    group_cols = [
        'run_id', 'evaluator_backend_label', 'evaluator_backend_model', 'subset_policy',
        'target_split', 'target_model', 'evaluator_variant', 'variant_role'
    ]
    for keys, group in cale_like.groupby(group_cols, dropna=False):
        run_id, backend_label, backend_model, subset_policy, target_split, target_model, variant, variant_role = keys
        row = {
            'run_id': run_id,
            'evaluator_backend_label': backend_label,
            'evaluator_backend_model': backend_model,
            'subset_policy': subset_policy,
            'target_split': target_split,
            'target_model': target_model,
            'evaluator_variant': variant,
            'variant_role': variant_role,
            'rows': len(group),
        }
        for col in cols:
            row[col] = pd.to_numeric(group[col], errors='coerce').mean()
        profile_rows.append(row)

construct_profile = pd.DataFrame(profile_rows)
construct_profile.to_csv(OUTDIR / 'construct_profile_by_run_backend_target_split_variant.csv', index=False)
construct_profile.to_csv(OUTDIR / 'construct_profile_backend_target_variant.csv', index=False)
construct_profile.head(20)


In [ ]:
def local_split_type(split):
    if split == 'pooled_qwen_llama':
        return 'balanced_pooled'
    if split == 'pooled_all_available_targets':
        return 'available_only_pooled'
    if str(split).startswith('target_'):
        return 'target_specific'
    return 'other'


def local_joined_unique(series):
    return ', '.join(sorted(series.dropna().astype(str).unique()))


def local_scope_frames(group):
    frames = []
    observed_splits = set(group['target_split'].dropna().astype(str)) if 'target_split' in group.columns else set()
    if EXPECTED_TARGET_SPLITS <= observed_splits:
        frames.append(('pooled_qwen_llama', group.copy()))
    else:
        frames.append(('pooled_all_available_targets', group.copy()))
    if 'target_split' in group.columns:
        for split_name, split_df in group.groupby('target_split', dropna=False):
            frames.append((split_name, split_df.copy()))
    return frames


full_profile_rows = []
full_cale_construct_profile = pd.DataFrame()
full_cale_construct_profile_compact = pd.DataFrame()
full_cale_target_delta = pd.DataFrame()

if not combined.empty:
    full_combined = combined[combined['evaluator_variant'].eq(FULL_CALE_VARIANT)].copy()
    full_cols = existing_behavior_columns(full_combined)
    if not full_combined.empty and full_cols:
        group_cols = ['evaluator_backend_family', 'evaluator_backend_label', 'evaluator_backend_model', 'evaluator_variant', 'variant_role']
        for keys, group in full_combined.groupby(group_cols, dropna=False):
            backend_family, backend_label, backend_model, variant, variant_role = keys
            for target_scope, split_df in local_scope_frames(group):
                row = {
                    'evaluator_backend_family': backend_family,
                    'evaluator_backend_label': backend_label,
                    'evaluator_backend_model': backend_model,
                    'evaluator_variant': variant,
                    'variant_role': variant_role,
                    'target_split': target_scope,
                    'split_type': local_split_type(target_scope),
                    'source_run_ids': local_joined_unique(split_df['run_id']) if 'run_id' in split_df.columns else '',
                    'target_models': local_joined_unique(split_df['target_model']) if 'target_model' in split_df.columns else '',
                    'rows': len(split_df),
                }
                for col in full_cols:
                    row[col] = pd.to_numeric(split_df[col], errors='coerce').mean()
                full_profile_rows.append(row)

        full_cale_construct_profile = pd.DataFrame(full_profile_rows)
        full_cale_construct_profile.to_csv(OUTDIR / 'full_cale_construct_profile_by_backend_protocol_target_split.csv', index=False)
        full_cale_construct_profile.to_csv(OUTDIR / 'full_cale_construct_profile_by_run_backend_target_split.csv', index=False)

        display_metrics = ['rows'] + [c for c in CORE_BEHAVIOR_COLUMNS if c in full_cale_construct_profile.columns]
        full_cale_construct_profile_compact = full_cale_construct_profile.pivot_table(
            index=['evaluator_backend_label', 'evaluator_backend_model', 'evaluator_variant'],
            columns='target_split',
            values=display_metrics,
            aggfunc='first',
        )
        # Put target_split as the outer column level so the display reads as
        # pooled/Qwen-target/Llama-target blocks rather than metric-first blocks.
        full_cale_construct_profile_compact = full_cale_construct_profile_compact.swaplevel(0, 1, axis=1)
        ordered_splits = [c for c in ['pooled_qwen_llama', 'target_qwen_only', 'target_llama_only', 'pooled_all_available_targets'] if c in full_cale_construct_profile_compact.columns.get_level_values(0)]
        if ordered_splits:
            full_cale_construct_profile_compact = full_cale_construct_profile_compact.reindex(ordered_splits, level=0, axis=1)
        full_cale_construct_profile_compact = full_cale_construct_profile_compact.sort_index(axis=1, level=[0, 1], sort_remaining=False)
        full_cale_construct_profile_compact.to_csv(OUTDIR / 'full_cale_construct_profile_compact.csv')
        display(full_cale_construct_profile_compact.round(3))

        heatmap_df = full_cale_construct_profile.copy()
        heatmap_df['row_label'] = heatmap_df['evaluator_backend_label'] + ' | ' + heatmap_df['evaluator_variant'] + ' | ' + heatmap_df['target_split'] + ' (n=' + heatmap_df['rows'].astype(str) + ')'
        heatmap_matrix = heatmap_df.set_index('row_label')[[c for c in full_cols if c in heatmap_df.columns]]
        save_heatmap(heatmap_matrix, OUTDIR / 'full_cale_construct_profile_backend_protocol_target_split_heatmap.png', 'Full CALE Construct Profile by Backend, Protocol, and Target Split')

        delta_rows = []
        metric_cols = [c for c in full_cols if c in full_cale_construct_profile.columns]
        for keys, group in full_cale_construct_profile.groupby(['evaluator_backend_label', 'evaluator_backend_model', 'evaluator_variant'], dropna=False):
            backend_label, backend_model, variant = keys
            by_split = group.set_index('target_split')
            if {'target_qwen_only', 'target_llama_only'} <= set(by_split.index):
                row = {
                    'evaluator_backend_label': backend_label,
                    'evaluator_backend_model': backend_model,
                    'evaluator_variant': variant,
                    'comparison': 'target_qwen_only_minus_target_llama_only',
                    'qwen_rows': int(by_split.loc['target_qwen_only', 'rows']),
                    'llama_rows': int(by_split.loc['target_llama_only', 'rows']),
                }
                for col in metric_cols:
                    row[col] = by_split.loc['target_qwen_only', col] - by_split.loc['target_llama_only', col]
                delta_rows.append(row)
        full_cale_target_delta = pd.DataFrame(delta_rows)
        full_cale_target_delta.to_csv(OUTDIR / 'full_cale_construct_profile_qwen_minus_llama.csv', index=False)
        if not full_cale_target_delta.empty:
            delta_heatmap = full_cale_target_delta.copy()
            delta_heatmap['row_label'] = delta_heatmap['evaluator_backend_label'] + ' | ' + delta_heatmap['evaluator_variant']
            delta_heatmap = delta_heatmap.set_index('row_label')[[c for c in metric_cols if c in delta_heatmap.columns]]
            save_heatmap(delta_heatmap, OUTDIR / 'full_cale_construct_profile_qwen_minus_llama_heatmap.png', 'Full CALE Construct Profile: Qwen Target Minus Llama Target')
else:
    print('No combined matrix loaded yet.')


## 9. Backend Alignment On Shared Rows

This checks whether different evaluator backends agree on the same target response and evaluator variant. It is backend robustness, not target-model performance.

这里看不同 evaluator backend 对同一条 target response、同一种 evaluator variant 是否一致。

In [ ]:
alignment_rows = []
if not combined.empty:
    sc = score_column(combined)
    if sc and 'id' in combined.columns:
        align_cols = ['run_id', 'evaluator_backend_label', 'target_model', 'target_split', 'id', 'evaluator_variant', sc]
        if 'dataset' in combined.columns:
            align_cols.insert(2, 'dataset')
        align = combined[align_cols].copy()
        align[sc] = pd.to_numeric(align[sc], errors='coerce')

        scopes = [('pooled_all_available_targets', align)]
        if 'target_split' in align.columns:
            observed_splits = set(align['target_split'].dropna().astype(str))
            if EXPECTED_TARGET_SPLITS <= observed_splits:
                scopes[0] = ('pooled_qwen_llama', align)
            scopes.extend(list(align.groupby('target_split', dropna=False)))

        for target_scope, scope_df in scopes:
            index_cols = ['target_model', 'id', 'evaluator_variant']
            if 'dataset' in scope_df.columns:
                index_cols.insert(1, 'dataset')
            for variant, variant_df in scope_df.groupby('evaluator_variant', dropna=False):
                wide = variant_df.pivot_table(index=index_cols, columns='evaluator_backend_label', values=sc, aggfunc='mean')
                for left, right in itertools.combinations(wide.columns, 2):
                    pair = wide[[left, right]].dropna()
                    if len(pair) < 3:
                        continue
                    alignment_rows.append({
                        'target_split': target_scope,
                        'evaluator_variant': variant,
                        'backend_left': left,
                        'backend_right': right,
                        'n_shared_rows': len(pair),
                        'pearson': pair[left].corr(pair[right], method='pearson'),
                        'spearman': pair[left].corr(pair[right], method='spearman'),
                        'mae': (pair[left] - pair[right]).abs().mean(),
                        'mean_signed_difference_left_minus_right': (pair[left] - pair[right]).mean(),
                        'index_columns': ', '.join(index_cols),
                    })

backend_alignment = pd.DataFrame(alignment_rows)
backend_alignment.to_csv(OUTDIR / 'backend_pair_score_alignment.csv', index=False)
backend_alignment.to_csv(OUTDIR / 'backend_pair_score_alignment_by_target_split.csv', index=False)
backend_alignment


## 10. Target-Model Dominance Check

For each evaluator backend, this section combines all available matched run files, then reruns PCA on `full_attack_aware_cale` rows for pooled Qwen+Llama targets and for each target split. If a backend only has one target split, the notebook labels that as available-only evidence instead of pretending it is balanced pooled evidence.

这里会先按 evaluator backend 合并可用的 matched run files，再只用 `full_attack_aware_cale` 做 PCA。只有同时有 Qwen 和 Llama target 时才显示 `pooled_qwen_llama`；否则不会假装它是 balanced pooled evidence。


In [ ]:
pca_rows = []
pca_loading_rows = []

if not combined.empty and 'evaluator_variant' in combined.columns:
    full_all = combined[combined['evaluator_variant'].eq(FULL_CALE_VARIANT)].copy()
else:
    full_all = pd.DataFrame()

if not full_all.empty:
    group_cols = ['evaluator_backend_label', 'evaluator_backend_model']
    for keys, backend_full in full_all.groupby(group_cols, dropna=False):
        backend_label, backend_model = keys
        run_ids = ', '.join(sorted(backend_full['run_id'].dropna().astype(str).unique())) if 'run_id' in backend_full.columns else ''
        split_frames = []
        observed_splits = set(backend_full['target_split'].dropna().astype(str)) if 'target_split' in backend_full.columns else set()
        if EXPECTED_TARGET_SPLITS <= observed_splits:
            split_frames.append(('pooled_qwen_llama', backend_full.copy()))
        else:
            split_frames.append(('pooled_all_available_targets', backend_full.copy()))
        for split_name, split_df in backend_full.groupby('target_split', dropna=False):
            split_frames.append((split_name, split_df.copy()))

        for split_name, split_df in split_frames:
            if len(split_df) < 5:
                pca_rows.append({
                    'evaluator_backend_label': backend_label,
                    'evaluator_backend_model': backend_model,
                    'source_run_ids': run_ids,
                    'target_split': split_name,
                    'rows': len(split_df),
                    'error': 'too few rows for PCA',
                })
                continue
            try:
                row, loadings, variance, scores = run_pca_summary(split_df, run_ids, backend_label, split_name)
            except ValueError as exc:
                pca_rows.append({
                    'evaluator_backend_label': backend_label,
                    'evaluator_backend_model': backend_model,
                    'source_run_ids': run_ids,
                    'target_split': split_name,
                    'rows': len(split_df),
                    'error': str(exc),
                })
                continue
            row['evaluator_backend_model'] = backend_model
            row['source_run_ids'] = run_ids
            pca_rows.append(row)
            safe_backend = str(backend_model).replace('/', '_').replace(' ', '_').replace(':', '_')
            safe_split = str(split_name).replace('/', '_').replace(' ', '_')
            loadings.to_csv(OUTDIR / f'{safe_backend}_{safe_split}_full_cale_pca_loadings.csv')
            variance.to_csv(OUTDIR / f'{safe_backend}_{safe_split}_full_cale_pca_explained_variance.csv', index=False)
            pca_loading_rows.extend(top_loading_records(loadings, run_ids, backend_label, split_name))

target_model_dominance_pca = pd.DataFrame(pca_rows)
target_model_dominance_loadings = pd.DataFrame(pca_loading_rows)
target_model_dominance_pca.to_csv(OUTDIR / 'target_model_dominance_pca_summary.csv', index=False)
target_model_dominance_loadings.to_csv(OUTDIR / 'target_model_dominance_pca_top_loadings.csv', index=False)
target_model_dominance_pca


In [ ]:
if not target_model_dominance_pca.empty and 'pc1_pc4_cumulative_variance' in target_model_dominance_pca.columns:
    dominance_var = target_model_dominance_pca.pivot_table(
        index=['evaluator_backend_label', 'evaluator_backend_model', 'source_run_ids'],
        columns='target_split',
        values='pc1_pc4_cumulative_variance',
        aggfunc='first',
    )
    dominance_var.to_csv(OUTDIR / 'target_model_dominance_variance_pivot.csv')
    save_annotated_heatmap(dominance_var, OUTDIR / 'target_model_dominance_variance.png', 'PC1-PC4 Variance by Backend and Target-Model Split', vmin=0, vmax=1)
    display(dominance_var.round(3))

    dominance_pc1 = target_model_dominance_pca.pivot_table(
        index=['evaluator_backend_label', 'evaluator_backend_model', 'source_run_ids'],
        columns='target_split',
        values='pc1_variance',
        aggfunc='first',
    )
    dominance_pc1.to_csv(OUTDIR / 'target_model_dominance_pc1_variance_pivot.csv')
    save_annotated_heatmap(dominance_pc1, OUTDIR / 'target_model_dominance_pc1_variance.png', 'PC1 Variance by Backend and Target-Model Split', vmin=0, vmax=1)
    display(dominance_pc1.round(3))


## 11. Protocol-Level Latent-Structure Audit

This section brings the weaker/control evaluator protocols into the same latent-structure audit. It does **not** ask which evaluator is best. It asks whether behavior variables become more or less low-dimensional under different scoring protocols and backends.

这里把之前差一点的 evaluator/protocol 也纳入同一个 PCA audit：heuristic 的 baseline/direct/generic/attack/full，以及 Qwen7B/DeepSeek 的 direct/full。它不是排行榜，而是在看不同 backend/protocol 下，behavior matrix 的 latent structure 是否稳定、是否过度单轴化。

Interpretation guardrails:

- High PC1 variance means the measured behavior variables are more concentrated on one dominant axis; it does not automatically mean the evaluator is better.
- PC1 is refit separately for every backend/protocol/split, so compare variance concentration and top-loading variable clusters, not raw PC signs.
- The heuristic full run has much more data than the strong-evaluator `limit1000` subsets; use `rows` and `subset_policy` when writing claims.

解释边界：PC1 高不等于 evaluator 更好；不同组的 PC1 是分别 fit 的；heuristic 是 full sample，strong evaluator 是 limit1000 subset，写结论时必须带上这个限制。


In [ ]:
VARIANT_ORDER = {
    'baseline_binary': 0,
    'baseline_likert': 1,
    'direct_trustllm_heuristic': 2,
    'direct_llm_judge': 2,
    'generic_cale': 3,
    'attack_aware_cale': 4,
    'full_attack_aware_cale': 5,
}

BACKEND_COLORS = {
    'Rule-based scoring backend (non-LLM)': '#7a7a7a',
    'Qwen2.5-7B local HF judge': '#1f77b4',
    'DeepSeek V4-Pro API judge': '#2ca02c',
}


def backend_short(label):
    value = str(label)
    if 'DeepSeek' in value:
        return 'DeepSeek V4-Pro'
    if 'Qwen2.5-7B' in value:
        return 'Qwen2.5-7B'
    if 'Rule-based' in value:
        return 'Rule-based heuristic'
    return value


def split_type(split):
    if split == 'pooled_qwen_llama':
        return 'balanced_pooled'
    if split == 'pooled_all_available_targets':
        return 'available_only_pooled'
    if str(split).startswith('target_'):
        return 'target_specific'
    return 'other'


def protocol_scope_frames(group):
    frames = []
    observed_splits = set(group['target_split'].dropna().astype(str)) if 'target_split' in group.columns else set()
    if EXPECTED_TARGET_SPLITS <= observed_splits:
        frames.append(('pooled_qwen_llama', group.copy()))
    else:
        frames.append(('pooled_all_available_targets', group.copy()))
    if 'target_split' in group.columns:
        for split_name, split_df in group.groupby('target_split', dropna=False):
            frames.append((split_name, split_df.copy()))
    return frames


def joined_unique(series):
    return ', '.join(sorted(series.dropna().astype(str).unique()))


protocol_pca_rows = []
protocol_pca_loading_rows = []

if not combined.empty:
    # Combine matched files by backend + protocol before PCA. This lets Qwen-target
    # and Llama-target limit1000 strong-evaluator runs form a real pooled split.
    group_cols = ['evaluator_backend_label', 'evaluator_backend_model', 'evaluator_variant', 'variant_role']
    for keys, group in combined.groupby(group_cols, dropna=False):
        backend_label, backend_model, variant, variant_role = keys
        source_run_ids = joined_unique(group['run_id']) if 'run_id' in group.columns else ''
        subset_policies = joined_unique(group['subset_policy']) if 'subset_policy' in group.columns else ''
        for split_name, split_df in protocol_scope_frames(group):
            row_base = {
                'run_id': source_run_ids,
                'source_run_ids': source_run_ids,
                'evaluator_backend_label': backend_label,
                'evaluator_backend_model': backend_model,
                'backend_short': backend_short(backend_label),
                'evaluator_variant': variant,
                'variant_role': variant_role,
                'variant_order': VARIANT_ORDER.get(str(variant), 99),
                'target_split': split_name,
                'split_type': split_type(split_name),
                'rows': len(split_df),
                'subset_policy': subset_policies,
            }
            if len(split_df) < 5:
                protocol_pca_rows.append({**row_base, 'error': 'too few rows for PCA'})
                continue
            try:
                row, loadings, variance, scores = run_pca_summary(split_df, source_run_ids, backend_label, split_name)
            except ValueError as exc:
                protocol_pca_rows.append({**row_base, 'error': str(exc)})
                continue
            row.update(row_base)
            protocol_pca_rows.append(row)
            for rec in top_loading_records(loadings, source_run_ids, backend_label, split_name):
                rec['source_run_ids'] = source_run_ids
                rec['evaluator_backend_model'] = backend_model
                rec['backend_short'] = backend_short(backend_label)
                rec['evaluator_variant'] = variant
                rec['variant_role'] = variant_role
                protocol_pca_loading_rows.append(rec)

protocol_backend_pca = pd.DataFrame(protocol_pca_rows)
protocol_backend_pca_loadings = pd.DataFrame(protocol_pca_loading_rows)
protocol_backend_pca.to_csv(OUTDIR / 'protocol_backend_pca_summary.csv', index=False)
protocol_backend_pca_loadings.to_csv(OUTDIR / 'protocol_backend_pca_top_loadings.csv', index=False)

if not protocol_backend_pca.empty:
    display_cols = [
        'backend_short', 'evaluator_variant', 'variant_role', 'target_split', 'split_type', 'rows',
        'n_columns_used', 'pc1_variance', 'pc1_pc4_cumulative_variance', 'pc1_top_variables', 'source_run_ids', 'error'
    ]
    sort_cols = [c for c in ['backend_short', 'variant_order', 'target_split'] if c in protocol_backend_pca.columns]
    display_df = protocol_backend_pca.sort_values(sort_cols) if sort_cols else protocol_backend_pca.copy()
    display(display_df[[c for c in display_cols if c in display_df.columns]].round(3))
else:
    print('No protocol-level PCA rows available.')


In [ ]:
def save_protocol_variance_plot(summary, path):
    if summary.empty:
        return
    plot_df = summary[
        summary['target_split'].isin(['pooled_qwen_llama', 'pooled_all_available_targets'])
        & summary.get('pc1_variance', pd.Series(index=summary.index, dtype=float)).notna()
    ].copy()
    if plot_df.empty:
        return
    plot_df = plot_df.sort_values(['backend_short', 'variant_order', 'evaluator_variant']).reset_index(drop=True)
    plot_df['audit_label'] = plot_df['backend_short'].map(short_backend_label) + ' | ' + plot_df['evaluator_variant'].map(short_variant_label)
    plot_df['row_label'] = plot_df['audit_label'] + ' (n=' + plot_df['rows'].astype(str) + ')'
    colors = plot_df['evaluator_backend_label'].map(BACKEND_COLORS).fillna('#4c4c4c')
    y = np.arange(len(plot_df))

    fig, axes = plt.subplots(1, 2, figsize=(13, max(4.5, 0.42 * len(plot_df))))
    metrics = [
        ('pc1_variance', 'PC1 explained variance'),
        ('pc1_pc4_cumulative_variance', 'PC1-PC4 cumulative variance'),
    ]
    for ax, (metric, title) in zip(axes, metrics):
        ax.barh(y, plot_df[metric], color=colors)
        ax.set_title(title)
        ax.set_xlim(0, 1)
        ax.set_xlabel('Explained variance ratio')
        ax.set_yticks(y)
        ax.set_yticklabels(plot_df['row_label'] if ax is axes[0] else [''] * len(plot_df), fontsize=8)
        ax.grid(axis='x', alpha=0.25)
        for idx, value in enumerate(plot_df[metric]):
            if pd.notna(value):
                ax.text(min(value + 0.015, 0.98), idx, f'{value:.3f}', va='center', fontsize=8)
    fig.suptitle('Latent-Structure Variance Across Backends and Protocols', y=1.01)
    fig.tight_layout()
    fig.savefig(path, dpi=240, bbox_inches='tight')
    plt.show()


def save_metric_by_target_plot(summary, metric, path, title):
    if summary.empty or metric not in summary.columns:
        return
    plot_df = summary[summary[metric].notna()].copy()
    if plot_df.empty:
        return
    plot_df['backend_target_label'] = plot_df['backend_short'].map(short_backend_label) + ' | ' + plot_df['evaluator_variant'].map(short_variant_label)
    pivot = plot_df.pivot_table(
        index=['backend_target_label'],
        columns='target_split',
        values=metric,
        aggfunc='first',
    )
    ordered_cols = [c for c in ['pooled_qwen_llama', 'target_qwen_only', 'target_llama_only', 'pooled_all_available_targets'] if c in pivot.columns]
    pivot = pivot[ordered_cols]
    pivot.to_csv(path.with_suffix('.csv'))
    save_annotated_heatmap(pivot, path, title, vmin=0, vmax=1)
    display(pivot.round(3))

if not protocol_backend_pca.empty:
    save_protocol_variance_plot(protocol_backend_pca, OUTDIR / 'protocol_backend_pca_variance_pooled.png')
    save_metric_by_target_plot(
        protocol_backend_pca,
        'pc1_variance',
        OUTDIR / 'protocol_backend_pca_pc1_by_target_split.png',
        'PC1 Variance by Backend, Protocol, and Target Split',
    )
    save_metric_by_target_plot(
        protocol_backend_pca,
        'pc1_pc4_cumulative_variance',
        OUTDIR / 'protocol_backend_pca_pc1_pc4_by_target_split.png',
        'PC1-PC4 Cumulative Variance by Backend, Protocol, and Target Split',
    )


## 12. CALE Variant Ladder Within The Heuristic Full Run

The heuristic full run is the only current result that contains the full CALE ladder (`generic_cale`, `attack_aware_cale`, `full_attack_aware_cale`) over both target models.

目前只有 heuristic full run 覆盖完整 CALE ladder，所以 variant ladder 的主分析应该放在这里。


In [ ]:
ladder_rows = []
heuristic = matrices.get('heuristic_full')
if heuristic is not None:
    cols = existing_behavior_columns(heuristic)
    ladder = heuristic[heuristic['evaluator_variant'].isin(CALE_VARIANTS)].copy()
    for keys, group in ladder.groupby(['target_split', 'evaluator_variant'], dropna=False):
        target_split, variant = keys
        row = {'target_split': target_split, 'evaluator_variant': variant, 'rows': len(group)}
        for col in cols:
            row[col] = pd.to_numeric(group[col], errors='coerce').mean()
        ladder_rows.append(row)

variant_ladder = pd.DataFrame(ladder_rows)
variant_ladder.to_csv(OUTDIR / 'heuristic_full_variant_ladder_by_target_split.csv', index=False)
if not variant_ladder.empty:
    display(variant_ladder.round(3))
else:
    print('Heuristic full result not available locally.')

### Paired CALE Family Ladder Delta

The heuristic full run is currently the only run with the complete CALE family ladder. This paired table asks whether `attack_aware_cale` and `full_attack_aware_cale` improve construct-relevant behavior over `generic_cale` on the same target responses.

目前只有 heuristic full run 有完整 CALE family ladder。这张 paired table 比较同一条 target response 上 `generic_cale → attack_aware_cale → full_attack_aware_cale` 的变化。


In [ ]:
paired_ladder_rows = []
heuristic = matrices.get('heuristic_full')
if heuristic is not None:
    ladder_variants = ['generic_cale', 'attack_aware_cale', 'full_attack_aware_cale']
    cols = existing_behavior_columns(heuristic)
    key_cols = ['dataset', 'id', 'target_model', 'target_split']
    ladder = heuristic[heuristic['evaluator_variant'].isin(ladder_variants)].copy()
    for metric in ['final_score'] + cols:
        if metric not in ladder.columns:
            continue
        tmp = ladder[key_cols + ['evaluator_variant', metric]].copy()
        tmp[metric] = pd.to_numeric(tmp[metric], errors='coerce')
        wide = tmp.pivot_table(index=key_cols, columns='evaluator_variant', values=metric, aggfunc='mean')
        comparisons = [
            ('attack_minus_generic', 'attack_aware_cale', 'generic_cale'),
            ('full_minus_attack', 'full_attack_aware_cale', 'attack_aware_cale'),
            ('full_minus_generic', 'full_attack_aware_cale', 'generic_cale'),
        ]
        for comparison, left, right in comparisons:
            if left not in wide.columns or right not in wide.columns:
                continue
            delta = (wide[left] - wide[right]).dropna()
            meta = wide.loc[delta.index].reset_index()[key_cols]
            for target_split, values in pd.DataFrame({'delta': delta.values, 'target_split': meta['target_split'].values}).groupby('target_split'):
                paired_ladder_rows.append({
                    'metric': metric,
                    'comparison': comparison,
                    'target_split': target_split,
                    'n_paired': len(values),
                    'mean_delta': values['delta'].mean(),
                    'median_delta': values['delta'].median(),
                    'std_delta': values['delta'].std(),
                })

paired_ladder_delta = pd.DataFrame(paired_ladder_rows)
paired_ladder_delta.to_csv(OUTDIR / 'heuristic_full_paired_cale_family_ladder_delta.csv', index=False)
paired_ladder_delta.head(40)


## 13. Proxy-Specific Diagnostics

Proxy variables are reference-label-specific. They should be interpreted only where the corresponding FEVER label applies, not as general scores across all rows.

Proxy 变量是 reference-label-specific，不能把非对应 label 的缺失当成 0。


In [ ]:
proxy_specs = {
    'NOT ENOUGH INFO': 'nei_uncertainty_failure_proxy',
    'REFUTES': 'refutes_correction_credit_proxy',
    'SUPPORTS': 'supports_status_failure_proxy',
}
proxy_rows = []
if not combined.empty and 'reference_label' in combined.columns:
    group_cols = [
        'run_id', 'evaluator_backend_label', 'evaluator_backend_model', 'subset_policy',
        'target_split', 'target_model', 'evaluator_variant'
    ]
    for label, proxy in proxy_specs.items():
        if proxy not in combined.columns:
            continue
        subset = combined[combined['reference_label'].eq(label)].copy()
        for keys, group in subset.groupby(group_cols, dropna=False):
            run_id, backend_label, backend_model, subset_policy, target_split, target_model, variant = keys
            proxy_rows.append({
                'run_id': run_id,
                'reference_label': label,
                'proxy': proxy,
                'evaluator_backend_label': backend_label,
                'evaluator_backend_model': backend_model,
                'subset_policy': subset_policy,
                'target_split': target_split,
                'target_model': target_model,
                'evaluator_variant': variant,
                'rows': len(group),
                'mean_proxy': pd.to_numeric(group[proxy], errors='coerce').mean(),
                'non_null_share': pd.to_numeric(group[proxy], errors='coerce').notna().mean(),
            })

proxy_diagnostics = pd.DataFrame(proxy_rows)
proxy_diagnostics.to_csv(OUTDIR / 'proxy_specific_diagnostics.csv', index=False)
proxy_diagnostics.to_csv(OUTDIR / 'proxy_specific_diagnostics_by_run_target_split.csv', index=False)
proxy_diagnostics.head(30)


## 14. Paper-Facing Global Summary

This final cell writes a compact markdown summary. Treat it as a draft note, not as final paper prose.

最后这一步生成一个简短 markdown summary，作为写作草稿，不是最终论文语言。


In [ ]:
summary_lines = []
summary_lines.append('# Global Evaluator Audit Summary')
summary_lines.append('')
summary_lines.append('Layer reminder: `target_model` is the response-generating model; `evaluator_backend_label` is the judge backend; `evaluator_variant` is the evaluation protocol.')
summary_lines.append('This is an audit notebook, not a leaderboard. We include weaker or simpler evaluator/protocol settings as robustness and sensitivity controls, not as interchangeable judges in a single calibrated ranking.')
summary_lines.append('Evaluator evidence is organized as backend/protocol/split, not as a flat four-model evaluator comparison.')
summary_lines.append('')

if not audit.empty:
    summary_lines.append('## File Audit')
    for _, row in audit.iterrows():
        summary_lines.append(
            f"- {row['evaluator_backend_label']} / {row['run_id']}: exists={row['exists']}, report_exists={row.get('report_path_exists', False)}, rows={row['rows']}/{row['expected_rows']}, "
            f"coverage={row['target_coverage']}, variants=[{row['evaluator_variants']}]."
        )
    summary_lines.append('')

if 'protocol_backend_pca' in globals() and not protocol_backend_pca.empty:
    summary_lines.append('## Protocol-Level PCA Audit')
    pooled = protocol_backend_pca[protocol_backend_pca['target_split'].isin(['pooled_qwen_llama', 'pooled_all_available_targets'])].copy()
    pooled = pooled[pooled.get('pc1_pc4_cumulative_variance', pd.Series(index=pooled.index, dtype=float)).notna()]
    for _, row in pooled.sort_values(['backend_short', 'variant_order', 'evaluator_variant']).iterrows():
        summary_lines.append(
            f"- {row['backend_short']} / {row['evaluator_variant']} / {row['target_split']}: "
            f"PC1={row.get('pc1_variance', np.nan):.3f}, PC1-PC4={row.get('pc1_pc4_cumulative_variance', np.nan):.3f}, "
            f"n={int(row['rows'])}, PC1 top={row.get('pc1_top_variables', '')}."
        )
    summary_lines.append('')

if 'target_model_dominance_pca' in globals() and not target_model_dominance_pca.empty:
    summary_lines.append('## Full-CALE PCA / Target Dominance')
    for _, row in target_model_dominance_pca.iterrows():
        if pd.notna(row.get('pc1_pc4_cumulative_variance', np.nan)):
            summary_lines.append(
                f"- {row['evaluator_backend_label']} / {row['target_split']}: "
                f"PC1={row.get('pc1_variance', np.nan):.3f}, PC1-PC4={row['pc1_pc4_cumulative_variance']:.3f}; "
                f"PC1 top variables={row.get('pc1_top_variables', '')}."
            )
    summary_lines.append('')

summary_lines.append('## Recommended Outputs')
summary_lines.append('- `global_file_audit.csv`')
summary_lines.append('- `cross_product_coverage_compact.csv`')
summary_lines.append('- `outcome_summary_three_layers.csv`')
summary_lines.append('- `direct_vs_full_cale_delta.csv`')
summary_lines.append('- `protocol_backend_pca_summary.csv`')
summary_lines.append('- `protocol_backend_pca_variance_pooled.png`')
summary_lines.append('- `protocol_backend_pca_pc1_by_target_split.png`')
summary_lines.append('- `protocol_backend_pca_pc1_pc4_by_target_split.png`')
summary_lines.append('- `full_cale_construct_profile_by_backend_protocol_target_split.csv`')
summary_lines.append('- `full_cale_construct_profile_compact.csv`')
summary_lines.append('- `backend_pair_score_alignment_by_target_split.csv`')
summary_lines.append('- `schema_missingness_by_run_target_split.csv`')
summary_lines.append('- `proxy_specific_diagnostics_by_run_target_split.csv`')
summary_lines.append('- `target_model_dominance_pca_summary.csv`')
summary_lines.append('- `target_model_dominance_pc1_variance.png`')
summary_lines.append('- `target_model_dominance_variance.png`')
summary_lines.append('- `heuristic_full_variant_ladder_by_target_split.csv`')
summary_lines.append('')

summary_lines.append('## Caveats')
summary_lines.append('- Strong evaluator `limit1000` runs are interpreted as balanced backend evidence only when both Qwen-target and Llama-target matched files are present.')
summary_lines.append('- The rule-based heuristic backend is not Qwen, not Llama, and not a Qwen/Llama average.')
summary_lines.append('- Do not interpret cross-backend absolute scores or PCA variance as a calibrated leaderboard.')
summary_lines.append('- High PC1 variance means stronger one-axis concentration, not automatically better measurement quality.')
summary_lines.append('- PCA signs are arbitrary; interpret loading magnitudes and variable clusters.')

summary_text = chr(10).join(summary_lines)
(OUTDIR / 'global_interpretation_notes.md').write_text(summary_text, encoding='utf-8')
print(summary_text)
